# Attrition Model

## Pipeline Overview

| Step | Description |
|------|-------------|
| **1. Setup** | Imports, constants, plot theme |
| **2. Data Generation** | 5000-employee IBM-calibrated synthetic dataset |
| **3. Persona Validation** | Verify realistic attrition spread (5% – 55%) |
| **4. Feature Engineering** | 15+ derived interaction features |
| **5. EDA** | Visual exploration of all key drivers |
| **6. Preprocessing** | Encoding, scaling, train/test split |
| **7. Model Training** | 5 models with 5-fold stratified CV |
| **8. Model Comparison** | ROC, metrics table, confusion matrix |
| **9. Feature Importance** | Top 25 drivers of attrition |
| **10. Scoring** | Risk categories for every employee |
| **11. Save Artifacts** | Model + encoders + metadata for Streamlit |



---
## 1. Imports & Setup

In [48]:
import warnings
warnings.filterwarnings("ignore")

# Core
import numpy as np
import pandas as pd
import pickle, json, os, shutil
from datetime import datetime

print("✅ Core imports OK")

✅ Core imports OK


In [49]:
# Scikit-learn
from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import (RandomForestClassifier,
                                       GradientBoostingClassifier,
                                       HistGradientBoostingClassifier)
from sklearn.metrics          import (roc_auc_score, f1_score, accuracy_score,
                                       precision_score, recall_score,
                                       classification_report, confusion_matrix,
                                       roc_curve, average_precision_score)
from sklearn.inspection       import permutation_importance

print("✅ Sklearn imports OK")
from sklearn.calibration     import CalibratedClassifierCV

✅ Sklearn imports OK


Compared multiple classification algorithms because attrition prediction is a binary classification problem, and different models capture patterns differently. I also used calibration because business teams often care about probability quality, not just class labels.

In [50]:
# Visualisation + dark theme
import matplotlib
matplotlib.use("Agg") ## It allows charts to be generated even in non-interactive environments like servers or deployment environments
import matplotlib.pyplot as plt
import seaborn as sns

BG, FG, PANEL = "#0f1117", "white", "#1a1d2e"
PALETTE = ["#7c6af7", "#f59e0b", "#10b981", "#ef4444", "#38bdf8"]

def ax_style(ax, fig):
    """Apply consistent dark theme to any axis."""
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=FG, labelsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor("#44446688")

os.makedirs("artifacts", exist_ok=True)
print("✅ Visualisation setup OK — artifacts/ folder ready")

✅ Visualisation setup OK — artifacts/ folder ready


---
## 2. 🏗️ Data Generation

### 2.1 Reference Tables
All constants — bands, departments, salaries, logit weights — are defined here so they're easy to audit and change without touching the generation logic.

In [51]:
# Level bands — P1A (entry) through P6 (C-suite)
BANDS     = ["P1A","P1B","P1C","P2A","P2B","P2C","P3A","P3B","P4","P5","P6"]
BAND_PROB = [0.10,  0.09,  0.08,  0.12,  0.10,  0.08,  0.12,  0.10,  0.11,  0.06,  0.04]
#These are the probabilities of each level appearing in the synthetic dataset.So for example, lower levels are more common and top levels are fewer.


BAND_SALARY = {           # Annual CTC mid-point (INR)
    "P1A": 480_000,  "P1B": 620_000,  "P1C": 780_000,
    "P2A": 980_000,  "P2B": 1_200_000, "P2C": 1_450_000,
    "P3A": 1_700_000, "P3B": 2_100_000,
    "P4":  2_700_000, "P5":  3_800_000, "P6":  5_500_000,
}
BAND_NUM    = {b: i+1 for i, b in enumerate(BANDS)}
BAND_LOGIT  = {           # Higher band = more stable = lower risk
    "P1A": 0.55, "P1B": 0.40, "P1C": 0.25,
    "P2A": 0.15, "P2B": 0.05, "P2C": 0.00,
    "P3A":-0.10, "P3B":-0.20,
    "P4": -0.30, "P5": -0.45, "P6": -0.60,
}

##Assigned a risk effect for each level:lower levels have positive values and higher levels have negative values

print("Bands defined:", BANDS)

Bands defined: ['P1A', 'P1B', 'P1C', 'P2A', 'P2B', 'P2C', 'P3A', 'P3B', 'P4', 'P5', 'P6']


In [52]:
# Departments, Locations, Shifts
DEPTS    = ["Engineering","Sales","Finance","HR","Marketing",
            "Operations","Legal","Product","QA","Infra"]
DEPT_W   = [0.25, 0.18, 0.08, 0.05, 0.09, 0.12, 0.04, 0.09, 0.06, 0.04] # Weighted probability of each dept being sampled.
DEPT_MKT = {                  # Market salary multiplier per dept
    "Engineering": 1.22, "Finance": 1.15, "Legal": 1.12, "Product": 1.10,
    "QA": 1.05, "Marketing": 0.98, "Infra": 0.97, "Sales": 0.95,
    "Operations": 0.90, "HR": 0.88,
} # This affects how underpaid someone might look relative to market.
DEPT_LOGIT = {                # Sales/Engineering = high demand market
    "Engineering": 0.30, "Sales": 0.40, "Finance": 0.05, "HR": -0.05,
    "Marketing": 0.15, "Operations": -0.05, "Legal": -0.15, "Product": 0.20,
    "QA": 0.10, "Infra": 0.00,
} # department-wise attrition risk adjustments.

LOCS     = ["Bengaluru","Mumbai","Hyderabad","Pune","Chennai","Delhi","WFH"]
LOC_W    = [0.28, 0.18, 0.15, 0.13, 0.10, 0.08, 0.08]
LOC_COST = {
    "Bengaluru": 1.15, "Mumbai": 1.20, "Delhi": 1.12,
    "Hyderabad": 1.05, "Pune": 1.05, "Chennai": 1.03, "WFH": 0.90,
}

SHIFTS      = ["Day","Night","Rotational","Flexible"]
SHIFT_W     = [0.48, 0.14, 0.22, 0.16]
SHIFT_LOGIT = {"Day": 0.0, "Flexible": -0.25, "Rotational": 0.50, "Night": 0.65}
# I modeled location cost pressure, department demand, and shift burden as business drivers of attrition. These are realistic organizational factors often associated with employee churn.
print("Departments:", DEPTS)
print("Locations  :", LOCS)

Departments: ['Engineering', 'Sales', 'Finance', 'HR', 'Marketing', 'Operations', 'Legal', 'Product', 'QA', 'Infra']
Locations  : ['Bengaluru', 'Mumbai', 'Hyderabad', 'Pune', 'Chennai', 'Delhi', 'WFH']


In [53]:
# Log-odds coefficients — grounded in IBM HR Analytics research
# Positive = increases attrition risk | Negative = retention factor
COEF = {
    "intercept"          : -2.8,   # baseline → mean ~20%
    "overtime_yes"       : +1.30,  # strongest single predictor
    "pay_gap_per_unit"   : +5.00,  # (1 - market_rate_ratio) * this
    "low_hike"           : +0.60,  # hike < 5%
    "high_hike"          : -0.40,  # hike > 15%
    "job_sat"            : {1:+0.90, 2:+0.30, 3:-0.20, 4:-0.80},
    "wlb"                : {1:+0.70, 2:+0.20, 3:-0.20, 4:-0.60},
    "env_sat"            : {1:+0.50, 2:+0.10, 3:-0.10, 4:-0.40},
    "mgr_rating"         : {1:+0.70, 2:+0.30, 3:0.00, 4:-0.30, 5:-0.60},
    "promo_stagnant_4y"  : +0.70,
    "promo_stagnant_7y"  : +0.40,  # stacked with 4y
    "recent_promotion"   : -0.80,
    "awards_2plus"       : -0.50,
    "recognitions_3plus" : -0.35,
    "travel_rare"        : +0.25,
    "travel_frequent"    : +0.80,
    "wfh_per_day"        : -0.12,  # each WFH day reduces risk
    "single"             : +0.45,
    "divorced"           : +0.20,
    "female"             : -0.10,
    "young_under28"      : +0.50,
    "senior_over52"      : +0.30,
    "leave_spike_30d"    : +0.50,
    "leave_spike_90d"    : +0.40,
    "new_joiner_1y"      : +0.70,
    "veteran_10y"        : -0.40,
    "job_hopper_5cos"    : +0.35,
    "long_commute"       : +0.35,
}

print(f"✅ Coefficient table — {len(COEF)} entries loaded")

✅ Coefficient table — 28 entries loaded


### 2.2 Feature Samplers
Split into three focused functions so each is easy to read and test independently.

In [54]:
def sample_demographics_and_career(N, rng):
    """Sample org structure, demographics and career features."""
    band     = rng.choice(BANDS, size=N, p=BAND_PROB)
    dept     = rng.choice(DEPTS, size=N, p=DEPT_W)
    location = rng.choice(LOCS,  size=N, p=LOC_W)
    shift    = rng.choice(SHIFTS, size=N, p=SHIFT_W)
    gender   = rng.choice(["Male","Female","Other"], size=N, p=[0.56, 0.41, 0.03])
    marital  = rng.choice(["Single","Married","Divorced"], size=N, p=[0.35, 0.52, 0.13])
    age      = np.clip(rng.normal(34, 8, N).astype(int), 22, 60)
    edu      = rng.choice(["Bachelor","Master","PhD","Diploma","Other"],
                           size=N, p=[0.45, 0.33, 0.08, 0.10, 0.04])
    band_num = np.array([BAND_NUM[b] for b in band])

    yrs_co   = np.clip(rng.exponential(5, N).astype(int), 0, 35) #in real life, many employees have shorter tenure, and fewer stay very long.
    yrs_role = np.clip(rng.uniform(0, yrs_co + 1, N).astype(int), 0, yrs_co)
    yrs_mgr  = np.clip(rng.uniform(0, yrs_role + 1, N).astype(int), 0, yrs_role)
    total_ex = np.clip((age - 22) + rng.integers(0, 3, N), 0, 40)

    # Promotion — P1 bands stagnate more
    promo_b   = np.clip(rng.exponential(2.5, N), 0, 10).astype(int)
    yrs_promo = np.where(band_num <= 3,
                         np.clip(promo_b + rng.integers(0, 4, N), 1, 12),
                         np.clip(promo_b, 0, 8)).astype(int)
    promos_3y = np.where(yrs_promo <= 1, rng.integers(1, 3, N),
                np.where(yrs_promo <= 3, rng.integers(0, 2, N), 0))

    return dict(band=band, dept=dept, location=location, shift=shift,
                gender=gender, marital=marital, age=age, edu=edu, band_num=band_num,
                yrs_co=yrs_co, yrs_role=yrs_role, yrs_mgr=yrs_mgr,
                total_ex=total_ex, yrs_promo=yrs_promo, promos_3y=promos_3y)

print("✅ sample_demographics_and_career() defined")

✅ sample_demographics_and_career() defined


In [55]:
def sample_compensation(dc, rng):
    """Derive salary & market rate from band / dept / location."""
    N = len(dc["band"])
    base   = np.array([BAND_SALARY[b] for b in dc["band"]], dtype=float)
    mkt    = base * np.array([DEPT_MKT[d] for d in dc["dept"]]) \
                  * np.array([LOC_COST[l] for l in dc["location"]])
    actual = base * rng.normal(1.0, 0.12, N) * rng.uniform(0.85, 1.10, N)
    ratio  = actual / mkt
#This is MarketRateRatio.Meaning: ratio < 1 → paid below market and ratio > 1 → paid above market.This is a very important attrition driver.
    return dict(
        salary=actual.astype(int),
        mkt_ratio=ratio.round(3),
        hike=np.clip(rng.normal(9, 4, N), 0, 30).round(1),
        bonus=np.clip(rng.normal(10, 5, N), 0, 40).round(1),
    )

print("✅ sample_compensation() defined")

✅ sample_compensation() defined


In [56]:
def sample_engagement_and_leaves(N, rng):
    """Sample satisfaction, performance, leave and work-pattern features."""
    job_sat = rng.choice([1,2,3,4], size=N, p=[0.10, 0.20, 0.40, 0.30])
    env_sat = rng.choice([1,2,3,4], size=N, p=[0.08, 0.20, 0.42, 0.30])
    rel_sat = rng.choice([1,2,3,4], size=N, p=[0.08, 0.15, 0.45, 0.32])
    wlb     = rng.choice([1,2,3,4], size=N, p=[0.12, 0.22, 0.38, 0.28])
    avg_sat = ((job_sat + env_sat + rel_sat + wlb) / 4.0).round(2)

    perf   = rng.choice([1,2,3,4], size=N, p=[0.05, 0.15, 0.55, 0.25])
    awards = np.clip(rng.poisson(0.8, N), 0, 5)
    recogn = np.clip(rng.poisson(1.2, N), 0, 8)
    train  = rng.integers(0, 7, N)
    mgr    = rng.choice([1,2,3,4,5], size=N, p=[0.05, 0.10, 0.30, 0.35, 0.20])
    team   = rng.integers(4, 25, N)
    dist   = rng.integers(1, 60, N)

    # Cascading leave windows
    l30  = np.clip(rng.poisson(1.2, N), 0, 10)
    l90  = np.clip(rng.poisson(3.5, N), l30, 15)
    l180 = np.clip(rng.poisson(7.0, N), l90,  25)
    l365 = np.clip(rng.poisson(14,  N), l180, 40)
    sick = np.clip(rng.poisson(2.0, N), 0, 12)
    
    # Keeping this as rolling window or else it will generate the bad synthetic data.

    overtime = rng.choice(["Yes","No"], size=N, p=[0.32, 0.68])
    travel   = rng.choice(["None","Rare","Frequent"], size=N, p=[0.30, 0.50, 0.20])
    num_cos  = np.clip(rng.integers(0, 8, N), 0, 9)

    return dict(
        job_sat=job_sat, env_sat=env_sat, rel_sat=rel_sat, wlb=wlb, avg_sat=avg_sat,
        perf=perf, awards=awards, recogn=recogn, train=train,
        mgr=mgr, team=team, dist=dist,
        l30=l30, l90=l90, l180=l180, l365=l365, sick=sick,
        overtime=overtime, travel=travel, num_cos=num_cos,
    )

print("✅ sample_engagement_and_leaves() defined")

✅ sample_engagement_and_leaves() defined


### 2.3 Attrition Probability — Calibrated Log-Odds

Logit is **clipped to (−2.94, 0.20)** before sigmoid → probability range **5% – 55%**.  
This reflects real-world inertia: even the most miserable employee has job-search friction.

In [57]:
def compute_logit(dc, comp, eng):
    """Build log-odds from the three feature dictionaries."""
    N     = len(dc["band"])
    logit = np.full(N, COEF["intercept"], dtype=float)
    #The intercept is the baseline attrition tendency.clip(-2, 3) prevents extreme values.

    # Compensation
    pay_gap = 1.0 - comp["mkt_ratio"]
    logit  += np.clip(pay_gap * COEF["pay_gap_per_unit"], -2.0, 3.0)
    logit  += np.where(comp["hike"] < 5,  COEF["low_hike"],  0)
    logit  += np.where(comp["hike"] > 15, COEF["high_hike"], 0)

    # Overtime
    logit  += np.where(eng["overtime"] == "Yes", COEF["overtime_yes"], 0)

    # Satisfaction
    logit  += np.vectorize(COEF["job_sat"].get)(eng["job_sat"])
    logit  += np.vectorize(COEF["wlb"].get)(eng["wlb"])
    logit  += np.vectorize(COEF["env_sat"].get)(eng["env_sat"])
#That means lower satisfaction increases probability of leaving.
    # Manager
    logit  += np.vectorize(COEF["mgr_rating"].get)(eng["mgr"])

    # Promotion & recognition
    logit  += np.where(dc["yrs_promo"] >= 4, COEF["promo_stagnant_4y"], 0)
    logit  += np.where(dc["yrs_promo"] >= 7, COEF["promo_stagnant_7y"], 0)
    logit  += np.where(dc["promos_3y"] >= 1, COEF["recent_promotion"],  0)
    logit  += np.where(eng["awards"]   >= 2, COEF["awards_2plus"],      0)
    logit  += np.where(eng["recogn"]   >= 3, COEF["recognitions_3plus"],0)
#This reflects retention drivers.
    # Travel & shift
    travel_map = {"None": 0, "Rare": COEF["travel_rare"], "Frequent": COEF["travel_frequent"]}
    logit  += np.vectorize(travel_map.get)(eng["travel"])
    logit  += np.vectorize(SHIFT_LOGIT.get)(dc["shift"])
#Frequent travel and difficult shifts increase risk.
    # WFH (WFH location = 5 days)
    wfh_days = np.where(dc["location"] == "WFH", 5, 0)
    logit  += wfh_days * COEF["wfh_per_day"]

    # Demographics
    logit  += np.where(dc["marital"] == "Single",   COEF["single"],        0)
    logit  += np.where(dc["marital"] == "Divorced",  COEF["divorced"],      0)
    logit  += np.where(dc["gender"]  == "Female",    COEF["female"],        0)
    logit  += np.where(dc["age"]     <  28,           COEF["young_under28"], 0)
    logit  += np.where(dc["age"]     >  52,           COEF["senior_over52"], 0)

    # Leave spikes
    logit  += np.where(eng["l30"] >= 3, COEF["leave_spike_30d"], 0)
    logit  += np.where(eng["l90"] >= 8, COEF["leave_spike_90d"], 0)
#Recent leave spikes add risk.
    # Tenure
    logit  += np.where(dc["yrs_co"] <= 1,  COEF["new_joiner_1y"], 0)
    logit  += np.where(dc["yrs_co"] >= 10, COEF["veteran_10y"],   0)

    # Band & dept
    logit  += np.array([BAND_LOGIT[b]  for b in dc["band"]])
    logit  += np.array([DEPT_LOGIT[d]  for d in dc["dept"]])

    # Job-hopping & commute
    logit  += np.where(eng["num_cos"] >= 5,  COEF["job_hopper_5cos"], 0)
    logit  += np.where(eng["dist"]    > 35,  COEF["long_commute"],    0)

    return logit

print("✅ compute_logit() defined")

✅ compute_logit() defined


### 2.4 Assemble the Full Dataset

In [58]:
def generate_microland_employees(N=5000, seed=42):
    """
    Full pipeline: sample → logit → sigmoid → binary label.
    Probability range: min ~5%  |  max ~55%  |  mean ~20%
    """
    rng  = np.random.default_rng(seed)
    dc   = sample_demographics_and_career(N, rng)
    comp = sample_compensation(dc, rng)
    eng  = sample_engagement_and_leaves(N, rng)

    wfh_days = np.where(dc["location"] == "WFH", 5,
                rng.choice([0,1,2,3], size=N, p=[0.40, 0.25, 0.25, 0.10]))

    logit = compute_logit(dc, comp, eng)
    logit += rng.normal(0, 0.4, N)        # realistic noise:Real-world employee decisions are not fully explained by observed variables.Noise adds realism.
    logit  = np.clip(logit, -2.94, 0.20)  # prob range: 5% – 55%
    prob   = 1 / (1 + np.exp(-logit)) #This is the sigmoid function.
    attr_b = rng.binomial(1, prob, N) #This generates actual attrition outcome

    return pd.DataFrame({
        "EmployeeID"          : [f"ML{10000+i:05d}" for i in range(N)],
        # Demographics
        "Age"                 : dc["age"],
        "Gender"              : dc["gender"],
        "MaritalStatus"       : dc["marital"],
        "Education"           : dc["edu"],
        # Org
        "LevelBand"           : dc["band"],
        "LevelBandNum"        : dc["band_num"],
        "Department"          : dc["dept"],
        "Location"            : dc["location"],
        "ShiftType"           : dc["shift"],
        # Tenure & career
        "YearsAtCompany"      : dc["yrs_co"],
        "YearsInCurrentRole"  : dc["yrs_role"],
        "YearsWithCurrManager": dc["yrs_mgr"],
        "TotalWorkingYears"   : dc["total_ex"],
        "YearsSinceLastPromo" : dc["yrs_promo"],
        "PromotionsLast3Yrs"  : dc["promos_3y"],
        # Compensation
        "Salary_Annual"       : comp["salary"],
        "MarketRateRatio"     : comp["mkt_ratio"],
        "HikePercentLast"     : comp["hike"],
        "BonusPct"            : comp["bonus"],
        # Manager & team
        "ManagerRating"       : eng["mgr"],
        "TeamSize"            : eng["team"],
        "DistanceFromHome"    : eng["dist"],
        # Satisfaction (1-4 scale)
        "JobSatisfaction"     : eng["job_sat"],
        "EnvSatisfaction"     : eng["env_sat"],
        "RelationshipSat"     : eng["rel_sat"],
        "WorkLifeBalance"     : eng["wlb"],
        "AvgSatisfaction"     : eng["avg_sat"],
        # Performance
        "PerformanceRating"   : eng["perf"],
        "AwardsReceived"      : eng["awards"],
        "RecognitionsReceived": eng["recogn"],
        "TrainingLastYear"    : eng["train"],
        # Leave (4 windows)
        "Leaves_30d"          : eng["l30"],
        "Leaves_90d"          : eng["l90"],
        "Leaves_180d"         : eng["l180"],
        "Leaves_365d"         : eng["l365"],
        "SickLeave_Annual"    : eng["sick"],
        # Work patterns
        "OverTime"            : eng["overtime"],
        "BusinessTravel"      : eng["travel"],
        "NumCompaniesWorked"  : eng["num_cos"],
        "WFH_DaysPerWeek"     : wfh_days,
        # Target
        "AttritionProb_True" : prob.round(4),
        "AttritionBinary"    : attr_b,
        "Attrition"          : np.where(attr_b == 1, "Yes", "No"),
    })

print("✅ generate_microland_employees() defined")

✅ generate_microland_employees() defined


In [59]:
df = generate_microland_employees(N=5000, seed=42)

print(f"Shape      : {df.shape}")
print(f"Columns    : {df.shape[1]}")
print(f"\nAttrition rate: {df['AttritionBinary'].mean():.1%}")
print(df["Attrition"].value_counts().to_string())
df.head(3)

Shape      : (5000, 44)
Columns    : 44

Attrition rate: 26.7%
No     3663
Yes    1337


,EmployeeID,Age,Gender,MaritalStatus,Education,LevelBand,LevelBandNum,Department,Location,ShiftType,...,Leaves_180d,Leaves_365d,SickLeave_Annual,OverTime,BusinessTravel,NumCompaniesWorked,WFH_DaysPerWeek,AttritionProb_True,AttritionBinary,Attrition
0,ML10000,22,Female,Divorced,PhD,P3B,8,Engineering,Pune,Day,...,5,9,3,Yes,Rare,7,0,0.5498,0,No
1,ML10001,27,Other,Divorced,Bachelor,P2B,5,Operations,Pune,Day,...,7,13,2,No,Rare,2,1,0.0502,0,No
2,ML10002,26,Male,Married,Master,P4,9,Operations,Bengaluru,Day,...,6,8,1,No,Rare,5,2,0.0551,0,No


---
## 3. ✅ Persona Validation

Verify the data generation is realistic **before** modelling. These checks should show a wide spread — not a flat 20%.

In [60]:
# Probability spread check
prob_stats = df["AttritionProb_True"].describe()
print("=== Attrition Probability Distribution ===")
print(f"Min    : {df['AttritionProb_True'].min():.1%}  ← target ≥5%")
print(f"Max    : {df['AttritionProb_True'].max():.1%}  ← target ≤55%")
print(f"Mean   : {df['AttritionProb_True'].mean():.1%}  ← target ~20%")
print(f"Median : {df['AttritionProb_True'].median():.1%}")
print(f"P10    : {df['AttritionProb_True'].quantile(0.10):.1%}")
print(f"P90    : {df['AttritionProb_True'].quantile(0.90):.1%}")

#Because when generating synthetic targets, you must verify that the distribution is believable.

=== Attrition Probability Distribution ===
Min    : 5.0%  ← target ≥5%
Max    : 55.0%  ← target ≤55%
Mean   : 27.6%  ← target ~20%
Median : 23.0%
P10    : 5.0%
P90    : 55.0%


In [61]:
# Low-risk persona: married WFH woman with promotion & awards
low_risk = (
    (df["Gender"]           == "Female")  &
    (df["MaritalStatus"]    == "Married") &
    (df["Location"]         == "WFH")     &
    (df["PromotionsLast3Yrs"] >= 1)       &
    (df["AwardsReceived"]   >= 1)         &
    (df["JobSatisfaction"]  >= 3)         &
    (df["OverTime"]         == "No")
)
print("=== LOW RISK PERSONA ===")
print("Married · WFH · Female · Promoted · Awards · High Sat · No OT")
print(f"Count             : {low_risk.sum()}")
print(f"True prob (mean)  : {df.loc[low_risk,'AttritionProb_True'].mean():.1%}")
print(f"Observed attrition: {df.loc[low_risk,'AttritionBinary'].mean():.1%}")

# This validates whether the synthetic logic behaves the way the business expects.

=== LOW RISK PERSONA ===
Married · WFH · Female · Promoted · Awards · High Sat · No OT
Count             : 8
True prob (mean)  : 11.3%
Observed attrition: 12.5%


In [62]:
# High-risk persona: single underpaid sales on night shift with OT
high_risk = (
    (df["MaritalStatus"]    == "Single")                    &
    (df["Department"]       == "Sales")                     &
    (df["MarketRateRatio"]  < 0.85)                         &
    (df["OverTime"]         == "Yes")                       &
    (df["ShiftType"].isin(["Night","Rotational"]))
)
print("=== 🔴 HIGH RISK PERSONA ===")
print("Single · Sales · Underpaid · Overtime · Night/Rotational")
print(f"Count             : {high_risk.sum()}")
print(f"True prob (mean)  : {df.loc[high_risk,'AttritionProb_True'].mean():.1%}")
print(f"Observed attrition: {df.loc[high_risk,'AttritionBinary'].mean():.1%}")

=== 🔴 HIGH RISK PERSONA ===
Single · Sales · Underpaid · Overtime · Night/Rotational
Count             : 13
True prob (mean)  : 47.6%
Observed attrition: 15.4%


Note: Even if probability = 47.6%, it does NOT mean 47.6% WILL leave.xample to understand

If 10 employees each have 50% probability:

expected leavers ≈ 5
but actual could be:
3
6
4
2 (rare but possible)

Because it’s random sampling.This output shows a high-risk employee segment with an average modeled attrition probability of 47.6%, driven by factors like underpayment, overtime, and night shifts. However, the observed attrition is only 15.4% because the sample size is small (13 employees), and the target variable was generated probabilistically using a binomial process. Due to randomness, observed outcomes may deviate from expected probability in small samples, but would converge closer to the true probability with larger data.

In [63]:
# Segment-level spread
print("=== Attrition Rate Spread by Segment ===")
for col in ["Department","LevelBand","Gender","MaritalStatus","ShiftType","OverTime"]:
    r = df.groupby(col)["AttritionBinary"].mean()
    print(f"  {col:22s}: {r.min():.1%} – {r.max():.1%}  (spread={r.max()-r.min():.1%})")

=== Attrition Rate Spread by Segment ===
  Department            : 17.0% – 34.4%  (spread=17.4%)
  LevelBand             : 18.3% – 33.3%  (spread=15.0%)
  Gender                : 25.5% – 26.8%  (spread=1.3%)
  MaritalStatus         : 25.1% – 28.8%  (spread=3.7%)
  ShiftType             : 23.2% – 30.5%  (spread=7.3%)
  OverTime              : 22.6% – 35.5%  (spread=12.9%)


---
## 4. 🔧 Feature Engineering

Interaction features capture HR insights that raw columns cannot — e.g. being a high performer with zero recognition (`RecognitionGap`), or doing overtime while already dissatisfied (`OT_LowSat`).

In [64]:
def engineer_features(df_in):
    df = df_in.copy()

    # Compensation health
    df["IsUnderpaid"]         = (df["MarketRateRatio"] < 0.90).astype(int)
    df["IsSeverelyUnderpaid"] = (df["MarketRateRatio"] < 0.80).astype(int)
    df["PayGapPct"]           = ((1 - df["MarketRateRatio"]) * 100).clip(-50, 50).round(1)

    # Career stagnation
    df["StagnationIndex"]     = (df["YearsSinceLastPromo"] * 2
                                  - df["PromotionsLast3Yrs"] * 3).clip(0, 20)
#Created a stagnation index to capture career growth friction rather than relying on a single raw promotion variable.
    # Recognition gap (high performer, low reward)
    df["RecognitionGap"]      = (df["PerformanceRating"]
                                  - df["AwardsReceived"].clip(0, 4)).clip(0, 4)
#This measures whether a high performer is not receiving recognition.
    # Leave spikes
    df["LeaveSpike30"]        = (df["Leaves_30d"] >= 3).astype(int)
    df["LeaveSpike90"]        = (df["Leaves_90d"] >= 8).astype(int)
    df["LeaveVelocity"]       = (df["Leaves_365d"]
                                  / df["YearsAtCompany"].clip(1, 35)).round(2)

    # Satisfaction composite
    df["SatComposite"]        = (df["JobSatisfaction"]  * 0.35
                                + df["WorkLifeBalance"]  * 0.30
                                + df["EnvSatisfaction"]  * 0.20
                                + df["RelationshipSat"]  * 0.15).round(2)
    df["SatBelow2"]           = (df["AvgSatisfaction"] < 2.0).astype(int)
#Combined multiple satisfaction dimensions into one weighted score.
    # Interaction flags
    df["OT_LowSat"]           = ((df["OverTime"] == "Yes")
                                  & (df["JobSatisfaction"] <= 2)).astype(int)
    df["WFH_SatInteraction"]  = df["WFH_DaysPerWeek"] * df["JobSatisfaction"]
    df["TravelDistance"]      = ((df["BusinessTravel"] == "Frequent")
                                  * df["DistanceFromHome"])
#I engineered interaction terms to model combined effects, such as overtime paired with low satisfaction, which may be more predictive than either variable alone.
    # Tenure flags
    df["IsNewJoiner"]         = (df["YearsAtCompany"] <= 1).astype(int)
    df["IsVeteran"]           = (df["YearsAtCompany"] >= 10).astype(int)
    df["CompanyLoyaltyScore"] = (df["YearsAtCompany"]
                                  / df["TotalWorkingYears"].clip(1, 40)).round(2)

    # Mobility & shift
    df["IsJobHopper"]         = (df["NumCompaniesWorked"] >= 4).astype(int)
    df["IsDisruptiveShift"]   = df["ShiftType"].isin(["Night","Rotational"]).astype(int)
    df["IsFullWFH"]           = (df["WFH_DaysPerWeek"] == 5).astype(int)
    df["IsMarried"]           = (df["MaritalStatus"] == "Married").astype(int)

    # Composite risk score (rule-based)
    df["CompositeRiskFlags"]  = (
        df["IsUnderpaid"]
        + (df["StagnationIndex"] > 4).astype(int)
        + (df["OverTime"] == "Yes").astype(int)
        + df["LeaveSpike30"]
        + df["IsDisruptiveShift"]
    )
    return df


df = engineer_features(df)
print(f"Total features: {df.shape[1]}")
engineered = ["StagnationIndex","RecognitionGap","SatComposite",
              "OT_LowSat","CompositeRiskFlags","LeaveVelocity"]
df[engineered].describe().round(2)

Total features: 65


,StagnationIndex,RecognitionGap,SatComposite,OT_LowSat,CompositeRiskFlags,LeaveVelocity
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,3.92,2.21,2.90,0.09,1.76,6.93
std,5.05,1.11,0.49,0.29,1.00,5.81
min,0.00,0.00,1.00,0.00,0.00,0.26
25%,0.00,1.00,2.60,0.00,1.00,2.00
50%,1.00,2.00,2.95,0.00,2.00,4.50
75%,8.00,3.00,3.25,0.00,2.00,12.00
max,20.00,4.00,4.00,1.00,5.00,28.00


---
## 5. 📊 Exploratory Data Analysis

### 5.1 Overall Attrition Distribution

In [65]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Attrition Overview", color=FG, fontsize=13)

ax = axes[0]; ax_style(ax, fig)
vc   = df["Attrition"].value_counts()
bars = ax.bar(vc.index, vc.values, color=["#10b981","#ef4444"], alpha=0.9, width=0.5)
for b, v in zip(bars, vc.values):
    ax.text(b.get_x()+b.get_width()/2, v+30,
            f"{v:,} ({v/len(df):.1%})", ha="center", color=FG, fontsize=10)
ax.set_title("Count", color=FG); ax.set_ylabel("Employees", color=FG)
ax.set_ylim(0, vc.max() * 1.2)

ax = axes[1]; ax_style(ax, fig)
ax.hist(df["AttritionProb_True"], bins=50, color="#7c6af7", alpha=0.85, edgecolor="none")
ax.axvline(df["AttritionProb_True"].mean(), color="#f59e0b", lw=2,
           label=f"Mean={df['AttritionProb_True'].mean():.1%}")
ax.set_xlabel("Attrition Probability", color=FG); ax.set_ylabel("Count", color=FG)
ax.set_title("True Probability Distribution", color=FG)
ax.legend(facecolor=PANEL, labelcolor=FG)

plt.tight_layout()
plt.savefig("artifacts/eda_overview.png", dpi=120, bbox_inches="tight")
plt.show()

### 5.2 Attrition by Categorical Features

In [66]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Attrition Rate by Key Features", color=FG, fontsize=14)

def bar_attrition(ax, col, title, rotate=0):
    ax_style(ax, fig)
    grp = df.groupby(col)["AttritionBinary"].mean().sort_values(ascending=False)
    colors = plt.cm.RdYlGn_r(grp.values / grp.values.max())
    bars = ax.bar(grp.index.astype(str), grp.values, color=colors, alpha=0.9)
    for b, v in zip(bars, grp.values):
        ax.text(b.get_x()+b.get_width()/2, v+0.004,
                f"{v:.1%}", ha="center", color=FG, fontsize=7)
    ax.set_title(title, color=FG, fontsize=10)
    ax.set_ylabel("Attrition Rate", color=FG)
    ax.tick_params(axis='x', rotation=rotate)

bar_attrition(axes[0,0], "Department",    "By Department",       rotate=30)
bar_attrition(axes[0,1], "ShiftType",     "By Shift Type")
bar_attrition(axes[0,2], "OverTime",      "By Overtime")
bar_attrition(axes[1,0], "MaritalStatus", "By Marital Status")
bar_attrition(axes[1,1], "BusinessTravel","By Travel Frequency")
bar_attrition(axes[1,2], "Gender",        "By Gender")

plt.tight_layout()
plt.savefig("artifacts/eda_categorical.png", dpi=120, bbox_inches="tight")
plt.show()

### 5.3 Level Band, Satisfaction & Promotions

In [67]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
BAND_ORDER = ["P1A","P1B","P1C","P2A","P2B","P2C","P3A","P3B","P4","P5","P6"]

ax = axes[0]; ax_style(ax, fig)
band_rate = df.groupby("LevelBand")["AttritionBinary"].mean().reindex(BAND_ORDER)
ax.bar(band_rate.index, band_rate.fillna(0).values,
       color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(BAND_ORDER))), alpha=0.9)
ax.set_title("By Level Band", color=FG); ax.set_ylabel("Rate", color=FG)
ax.tick_params(axis='x', rotation=30, colors=FG)

ax = axes[1]; ax_style(ax, fig)
sat_rate = df.groupby("JobSatisfaction")["AttritionBinary"].mean()
ax.bar(sat_rate.index.astype(str), sat_rate.values,
       color=["#ef4444","#f97316","#eab308","#22c55e"], alpha=0.9)
for x, v in enumerate(sat_rate.values):
    ax.text(x, v+0.005, f"{v:.1%}", ha="center", color=FG, fontsize=9)
ax.set_xlabel("Job Satisfaction (1=Low, 4=High)", color=FG)
ax.set_title("By Job Satisfaction", color=FG)

ax = axes[2]; ax_style(ax, fig)
promo_rate = df.groupby("PromotionsLast3Yrs")["AttritionBinary"].mean()
ax.bar(promo_rate.index.astype(str), promo_rate.values, color="#7c6af7", alpha=0.9)
for x, v in enumerate(promo_rate.values):
    ax.text(x, v+0.003, f"{v:.1%}", ha="center", color=FG, fontsize=9)
ax.set_xlabel("Promotions in Last 3 Years", color=FG)
ax.set_title("By Promotion Count", color=FG)

plt.tight_layout()
plt.savefig("artifacts/eda_band_sat.png", dpi=120, bbox_inches="tight")
plt.show()

### 5.4 Pay vs Market & WFH Effect

In [68]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]; ax_style(ax, fig)
for lbl, g in df.groupby("Attrition"):
    ax.hist(g["MarketRateRatio"], bins=40, alpha=0.65, density=True,
            label=lbl, color={"Yes":"#ef4444","No":"#10b981"}[lbl])
ax.axvline(1.0, color="white", ls="--", lw=1.5, label="Market Parity")
ax.set_xlabel("Market Rate Ratio  (<1 = underpaid)", color=FG)
ax.set_ylabel("Density", color=FG)
ax.set_title("Pay vs Market — Attrition vs Retained", color=FG)
ax.legend(facecolor=PANEL, labelcolor=FG)

ax = axes[1]; ax_style(ax, fig)
wfh_rate = df.groupby("WFH_DaysPerWeek")["AttritionBinary"].mean()
ax.plot(wfh_rate.index, wfh_rate.values, "o-", color="#10b981", lw=2.5, ms=8)
ax.fill_between(wfh_rate.index, wfh_rate.values, alpha=0.15, color="#10b981")
for x, y in zip(wfh_rate.index, wfh_rate.values):
    ax.text(x, y+0.005, f"{y:.1%}", ha="center", color=FG, fontsize=8)
ax.set_xlabel("WFH Days per Week", color=FG)
ax.set_ylabel("Attrition Rate", color=FG)
ax.set_title("Attrition Rate vs WFH Flexibility", color=FG)

plt.tight_layout()
plt.savefig("artifacts/eda_pay_wfh.png", dpi=120, bbox_inches="tight")
plt.show()

### 5.5 Heatmaps — Gender × Dept & LevelBand × Shift

In [69]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG)

pivot1 = df.pivot_table(values="AttritionBinary",
                         index="Department", columns="Gender", aggfunc="mean") * 100
sns.heatmap(pivot1, annot=True, fmt=".1f", cmap="YlOrRd",
            ax=axes[0], linewidths=0.5, annot_kws={"size":9})
axes[0].set_title("Attrition % — Gender × Department", color=FG, fontsize=11)
axes[0].tick_params(colors=FG)

pivot2 = df.pivot_table(values="AttritionBinary",
                         index="LevelBand", columns="ShiftType", aggfunc="mean") * 100
pivot2 = pivot2.reindex([b for b in BAND_ORDER if b in pivot2.index])
sns.heatmap(pivot2, annot=True, fmt=".1f", cmap="YlOrRd",
            ax=axes[1], linewidths=0.5, annot_kws={"size":9})
axes[1].set_title("Attrition % — Level Band × Shift Type", color=FG, fontsize=11)
axes[1].tick_params(colors=FG)

plt.tight_layout()
plt.savefig("artifacts/eda_heatmaps.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 6. ⚙️ Preprocessing

In [70]:
# Columns to exclude from the model
DROP_COLS = [
    "EmployeeID",
    "Attrition",
    "AttritionBinary",
    "AttritionProb_True",   # true prob — not available in production!
]

CATEGORICAL = [
    "Gender", "MaritalStatus", "Education", "LevelBand",
    "Department", "Location", "ShiftType", "OverTime", "BusinessTravel",
]

print(f"Dropping   : {DROP_COLS}")
print(f"Encoding   : {CATEGORICAL}")

Dropping   : ['EmployeeID', 'Attrition', 'AttritionBinary', 'AttritionProb_True']
Encoding   : ['Gender', 'MaritalStatus', 'Education', 'LevelBand', 'Department', 'Location', 'ShiftType', 'OverTime', 'BusinessTravel']


In [71]:
def preprocess(df_in, encoders=None, fit=True):
    """Label-encode categoricals. Returns (X, y, encoders)."""
    df       = df_in.copy()
    y        = df["AttritionBinary"].values
    X        = df[[c for c in df.columns if c not in DROP_COLS]].copy()
    encoders = encoders or {}

    for col in CATEGORICAL:
        if col not in X.columns:
            continue
        if fit:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            encoders[col] = le
        else:
            le = encoders[col]
            X[col] = X[col].astype(str).apply(
                lambda v: le.transform([v])[0] if v in le.classes_ else -1 #Unseen value handling as when new category appears later, it gets -1
            )
    return X, y, encoders
#I used label encoding across the pipeline, though in a production linear-model setup I would often prefer one-hot encoding to avoid imposing ordinal structure on nominal categories.

X, y, encoders = preprocess(df, fit=True)
feature_names  = list(X.columns)

print(f"Feature matrix : {X.shape}")
print(f"Class balance  : Stay={(y==0).sum()} | Leave={(y==1).sum()} | rate={y.mean():.3f}")

Feature matrix : (5000, 61)
Class balance  : Stay=3663 | Leave=1337 | rate=0.267


In [72]:
# 80 / 20 stratified split + scaling
X_tr, X_te, y_tr, y_te = train_test_split(
    X.values, y, test_size=0.20, random_state=42, stratify=y
)
#I used stratified splitting to preserve the attrition class ratio across train and test sets, which is important for stable evaluation in classification problems.
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)
cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#I fit the scaler only on training data and transformed test data using the learned parameters to prevent data leakage.
print(f"Train : {len(X_tr):,}  |  Test : {len(X_te):,}  |  CV : 5-fold Stratified")

Train : 4,000  |  Test : 1,000  |  CV : 5-fold Stratified


---
## 7. 🤖 Model Training

We keep the **same model family comparison** and the **same thresholding idea**.

The only targeted fix added here is:

- train candidate models as before
- pick the strongest base model
- **calibrate the final deployed model** so probabilities are less inflated and more realistic

This helps avoid too many employees clustering at `0.90+` predicted risk.


### 7.1 Define candidate models

In [73]:
MODELS = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced",
                           C=0.5, solver="lbfgs"),
        True    # needs scaling
    ),
    "Decision Tree": (
        DecisionTreeClassifier(max_depth=8, min_samples_leaf=20,
                               class_weight="balanced", random_state=42),
        False
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=300, max_depth=12,
                               min_samples_leaf=10, class_weight="balanced",
                               random_state=42, n_jobs=-1),
        False
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                   learning_rate=0.05, subsample=0.8,
                                   random_state=42),
        False
    ),
    "HistGradientBoosting": (
        HistGradientBoostingClassifier(max_iter=300, max_depth=6,
                                       learning_rate=0.05,
                                       class_weight="balanced",
                                       random_state=42),
        False
    ),
}

#I evaluated both interpretable and non-linear ensemble methods to balance explainability with predictive power.

print("Models:", list(MODELS.keys()))

Models: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'HistGradientBoosting']


### 7.2 Training settings

In [74]:
DECISION_THRESHOLD = 0.40   # I lowered the classification threshold from 0.50 to 0.40 to improve recall for likely leavers, since attrition interventions are often more sensitive to false negatives than false positives.)
results = {}

print(f"Training {len(MODELS)} models | CV=5-fold | threshold={DECISION_THRESHOLD}\n")
print(f"{'Model':25s} | {'CV AUC':>15s} | {'Test AUC':>9s} | {'F1':>6s} | {'Recall':>7s}")
print("-" * 75)


Training 5 models | CV=5-fold | threshold=0.4

Model                     |          CV AUC |  Test AUC |     F1 |  Recall
---------------------------------------------------------------------------


### 7.3 Train all base models

In [75]:
print(f"{'Model':25s} | {'CV AUC':>15s} | {'Test AUC':>9s} | {'F1':>6s} | {'Recall':>7s}")
print("-" * 75)
for name, (mdl, scale) in MODELS.items():
    Xtr, Xte = (X_tr_sc, X_te_sc) if scale else (X_tr, X_te)

    cv_s  = cross_val_score(mdl, Xtr, y_tr, cv=cv, scoring="roc_auc", n_jobs=-1) #AUC:Because attrition is a classification problem and AUC measures ranking quality across thresholds.
    mdl.fit(Xtr, y_tr)
    
    
    y_pr  = mdl.predict_proba(Xte)[:, 1]
    y_pd  = (y_pr >= DECISION_THRESHOLD).astype(int)

#Convert probability into predicted class using threshold 0.40.

    results[name] = dict(
        model=mdl,
        needs_scale=scale,
        cv_auc_mean=cv_s.mean(),
        cv_auc_std=cv_s.std(),
        test_auc=roc_auc_score(y_te, y_pr),
        test_f1=f1_score(y_te, y_pd),
        test_acc=accuracy_score(y_te, y_pd),
        test_prec=precision_score(y_te, y_pd, zero_division=0),
        test_recall=recall_score(y_te, y_pd),
        avg_prec=average_precision_score(y_te, y_pr),
        y_prob=y_pr,
        y_pred=y_pd,
    )
#Evaluated the models using both threshold-independent metrics like ROC-AUC and threshold-dependent metrics like F1, precision, and recall to get a balanced view of ranking quality and classification performance.
    r = results[name]
    print(f"{name:25s} | {r['cv_auc_mean']:.4f} ±{r['cv_auc_std']:.4f} "
          f"| {r['test_auc']:.4f}   | {r['test_f1']:.4f} | {r['test_recall']:.4f}")


Model                     |          CV AUC |  Test AUC |     F1 |  Recall
---------------------------------------------------------------------------
Logistic Regression       | 0.7575 ±0.0169 | 0.7470   | 0.5509 | 0.8315
Decision Tree             | 0.6622 ±0.0263 | 0.6593   | 0.4722 | 0.7303
Random Forest             | 0.7295 ±0.0159 | 0.7365   | 0.5370 | 0.7753
Gradient Boosting         | 0.7322 ±0.0146 | 0.7311   | 0.4471 | 0.4270
HistGradientBoosting      | 0.7230 ±0.0229 | 0.7256   | 0.5170 | 0.6255


### 7.4 Select the best base model

Selection logic is unchanged: we still choose the strongest base model by **CV AUC**.


In [76]:
best_name   = max(results, key=lambda k: results[k]["cv_auc_mean"])
best_model  = results[best_name]["model"]
needs_scale = results[best_name]["needs_scale"]

print(f"\n🏆 Best base model : {best_name}")
print(f"   CV AUC          : {results[best_name]['cv_auc_mean']:.4f}")
print(f"   Test AUC        : {results[best_name]['test_auc']:.4f}")
print(f"   F1 Score        : {results[best_name]['test_f1']:.4f}")
print(f"   Recall          : {results[best_name]['test_recall']:.4f}")



🏆 Best base model : Logistic Regression
   CV AUC          : 0.7575
   Test AUC        : 0.7470
   F1 Score        : 0.5509
   Recall          : 0.8315


### 7.5 Calibrate the deployed model

This is the **main fix** for your issue.

Tree-based models often rank employees well, but their raw `predict_proba()` values can be too extreme.  
So we calibrate only the final deployed model and keep the rest of the pipeline intact.


In [77]:
Xtr_best, Xte_best = (X_tr_sc, X_te_sc) if needs_scale else (X_tr, X_te)

calibrated_model = CalibratedClassifierCV(best_model, method="sigmoid", cv=5)
calibrated_model.fit(Xtr_best, y_tr)
#Since the output probabilities were intended for risk scoring and intervention prioritization, I calibrated the best base model so the predicted probabilities better aligned with observed event frequency.
cal_y_prob = calibrated_model.predict_proba(Xte_best)[:, 1]
cal_y_pred = (cal_y_prob >= DECISION_THRESHOLD).astype(int)

calibrated_results = {
    "test_auc": roc_auc_score(y_te, cal_y_prob),
    "test_f1": f1_score(y_te, cal_y_pred),
    "test_acc": accuracy_score(y_te, cal_y_pred),
    "test_prec": precision_score(y_te, cal_y_pred, zero_division=0),
    "test_recall": recall_score(y_te, cal_y_pred),
    "avg_prec": average_precision_score(y_te, cal_y_prob),
    "y_prob": cal_y_prob,
    "y_pred": cal_y_pred,
}

print("Calibrated deployed model metrics:")
for k in ["test_auc", "test_f1", "test_acc", "test_prec", "test_recall", "avg_prec"]:
    print(f"  {k:12s}: {calibrated_results[k]:.4f}")


Calibrated deployed model metrics:
  test_auc    : 0.7470
  test_f1     : 0.4487
  test_acc    : 0.7420
  test_prec   : 0.5224
  test_recall : 0.3933
  avg_prec    : 0.4786


### 7.6 Quick probability spread check

This check helps confirm that probabilities are no longer unnecessarily concentrated at the top end.


In [78]:
print("Raw model probability summary:")
print(pd.Series(results[best_name]["y_prob"]).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(4))

print("\nCalibrated model probability summary:")
print(pd.Series(calibrated_results["y_prob"]).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(4))


Raw model probability summary:
count    1000.0000
mean        0.4435
std         0.2385
min         0.0140
10%         0.1333
25%         0.2361
50%         0.4326
75%         0.6316
90%         0.7971
max         0.9684
dtype: float64

Calibrated model probability summary:
count    1000.0000
mean        0.2655
std         0.1718
min         0.0115
10%         0.0751
25%         0.1262
50%         0.2316
75%         0.3684
90%         0.5323
max         0.8582
dtype: float64


---
## 8. 📈 Model Comparison

### 8.1 Summary Table

In [79]:
summary = pd.DataFrame({
    name: {
        "CV AUC"        : f"{r['cv_auc_mean']:.4f} ±{r['cv_auc_std']:.4f}",
        "Test AUC"      : round(r["test_auc"], 4),
        "F1 Score"      : round(r["test_f1"], 4),
        "Precision"     : round(r["test_prec"], 4),
        "Recall"        : round(r["test_recall"], 4),
        "Accuracy"      : round(r["test_acc"], 4),
    }
    for name, r in results.items()
}).T.sort_values("Test AUC", ascending=False)

summary


,CV AUC,Test AUC,F1 Score,Precision,Recall,Accuracy
Logistic Regression,0.7575 ±0.0169,0.747,0.5509,0.4119,0.8315,0.638
Random Forest,0.7295 ±0.0159,0.7365,0.537,0.4107,0.7753,0.643
Gradient Boosting,0.7322 ±0.0146,0.7311,0.4471,0.4691,0.427,0.718
HistGradientBoosting,0.7230 ±0.0229,0.7256,0.517,0.4406,0.6255,0.688
Decision Tree,0.6622 ±0.0263,0.6593,0.4722,0.3488,0.7303,0.564


### 8.2 ROC Curves

In [80]:
fig, ax = plt.subplots(figsize=(7, 5))
ax_style(ax, fig)

for (name, res), c in zip(results.items(), PALETTE):
    fpr, tpr, _ = roc_curve(y_te, res["y_prob"])
    ax.plot(fpr, tpr, color=c, lw=2, label=f"{name.split()[0]} (AUC={res['test_auc']:.3f})")
ax.plot([0,1],[0,1], "w--", lw=0.8, alpha=0.4, label="Random")

ax.set_xlabel("False Positive Rate", color=FG)
ax.set_ylabel("True Positive Rate", color=FG)
ax.set_title("ROC Curves — Test Set", color=FG, fontsize=12)
ax.legend(facecolor=PANEL, labelcolor=FG, fontsize=9)

plt.tight_layout()
plt.savefig("artifacts/roc_curves.png", dpi=120, bbox_inches="tight")
plt.show()

#It visually compares sensitivity-specificity tradeoff across thresholds.

### 8.3 Metric Bar Comparison

In [81]:
fig, ax = plt.subplots(figsize=(12, 5))
ax_style(ax, fig)

names = list(results.keys())
x, w  = np.arange(len(names)), 0.20
ax.bar(x-1.5*w, [results[n]["cv_auc_mean"]  for n in names], w, color="#7c6af7", label="CV AUC",   alpha=0.9)
ax.bar(x-0.5*w, [results[n]["test_auc"]      for n in names], w, color="#10b981", label="Test AUC", alpha=0.9)
ax.bar(x+0.5*w, [results[n]["test_f1"]       for n in names], w, color="#f59e0b", label="F1",       alpha=0.9)
ax.bar(x+1.5*w, [results[n]["test_recall"]   for n in names], w, color="#ef4444", label="Recall",   alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([n.replace(" ","\n") for n in names], color=FG, fontsize=8)
ax.set_ylim(0, 1.1); ax.set_title("Model Performance Comparison", color=FG, fontsize=12)
ax.legend(facecolor=PANEL, labelcolor=FG)

plt.tight_layout()
plt.savefig("artifacts/model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

### 8.4 Confusion Matrix & Classification Report

In [82]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.patch.set_facecolor(BG)

# Raw best model confusion matrix
ax = axes[0]
ax_style(ax, fig)
cm_raw = confusion_matrix(y_te, results[best_name]["y_pred"])
sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            annot_kws={"color":"black", "fontsize":11})
ax.set_title(f"Raw Best Model — {best_name}", color=FG)
ax.set_xlabel("Predicted", color=FG)
ax.set_ylabel("Actual", color=FG)

# Calibrated deployed model confusion matrix
ax = axes[1]
ax_style(ax, fig)
cm_cal = confusion_matrix(y_te, calibrated_results["y_pred"])
sns.heatmap(cm_cal, annot=True, fmt="d", cmap="Greens", cbar=False, ax=ax,
            annot_kws={"color":"black", "fontsize":11})
ax.set_title("Calibrated Deployed Model", color=FG)
ax.set_xlabel("Predicted", color=FG)
ax.set_ylabel("Actual", color=FG)

plt.tight_layout()
plt.show()

print(f"=== Classification Report — Raw {best_name} ===")
print(classification_report(y_te, results[best_name]["y_pred"], target_names=['Stay','Leave']))

print(f"\n=== Classification Report — Calibrated {best_name} ===")
print(classification_report(y_te, calibrated_results["y_pred"], target_names=['Stay','Leave']))


=== Classification Report — Raw Logistic Regression ===
              precision    recall  f1-score   support

        Stay       0.90      0.57      0.70       733
       Leave       0.41      0.83      0.55       267

    accuracy                           0.64      1000
   macro avg       0.66      0.70      0.62      1000
weighted avg       0.77      0.64      0.66      1000


=== Classification Report — Calibrated Logistic Regression ===
              precision    recall  f1-score   support

        Stay       0.80      0.87      0.83       733
       Leave       0.52      0.39      0.45       267

    accuracy                           0.74      1000
   macro avg       0.66      0.63      0.64      1000
weighted avg       0.72      0.74      0.73      1000



---
## 9. 🔍 Feature Importance

In [83]:
Xte_imp = X_te_sc if needs_scale else X_te

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
else:
    r = permutation_importance(best_model, Xte_imp, y_te,
                               n_repeats=10, random_state=42)
    importances = r.importances_mean

imp_df = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
            .sort_values("Importance", ascending=False)
            .head(25))

print(f"Top 10 features driving {best_name}:")
print(imp_df.head(10).to_string(index=False))

Top 10 features driving Logistic Regression:
           Feature  Importance
         PayGapPct    0.355755
          OverTime    0.346122
    TravelDistance    0.282086
   MarketRateRatio    0.274479
 IsDisruptiveShift    0.265609
     ManagerRating    0.239097
       IsNewJoiner    0.233981
PromotionsLast3Yrs    0.198135
         IsVeteran    0.188367
   RelationshipSat    0.165398


I used model-appropriate feature importance extraction and fell back to permutation importance when native importance was unavailable, so I could still explain key drivers of attrition.

In [84]:
fig, ax = plt.subplots(figsize=(10, 9))
ax_style(ax, fig)
colors_imp = plt.cm.RdYlGn_r(np.linspace(0.1, 0.85, len(imp_df)))
ax.barh(imp_df["Feature"][::-1], imp_df["Importance"][::-1],
        color=colors_imp[::-1], alpha=0.9)
ax.set_title(f"Top 25 Feature Importances — {best_name}", color=FG, fontsize=12)
ax.set_xlabel("Importance", color=FG)
ax.tick_params(colors=FG, labelsize=8)
plt.tight_layout()
plt.savefig("artifacts/feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 10. Score All Employees

For deployment, we now score the full dataset using the **calibrated deployed model** instead of raw probabilities from the best base model.

Raw model outputs are not always well-calibrated probabilities, especially for tree-based models like Random Forest and Gradient Boosting. Calibration techniques like Platt Scaling or Isotonic Regression adjust predicted probabilities to better reflect true likelihoods. This is critical in business use cases like attrition prediction, where decisions depend on probability thresholds rather than just class labels.


In [85]:
def risk_cat(p):
    if p >= 0.45:
        return "Critical"
    if p >= 0.32:
        return "High"
    if p >= 0.18:
        return "Medium"
    return "Low"

In [86]:
X_full = X.values
X_pred = scaler.transform(X_full) if needs_scale else X_full

df["AttritionProb"] = calibrated_model.predict_proba(X_pred)[:, 1].round(4)
df["RiskCategory"]  = df["AttritionProb"].apply(risk_cat)
df_scored           = df.copy()

print("Risk category distribution:")
print(df_scored["RiskCategory"].value_counts())

print("\nAttrition rate & avg prob by risk category:")
print(df_scored.groupby("RiskCategory")[["AttritionBinary", "AttritionProb"]].mean().round(3))


Risk category distribution:
Low         1859
Medium      1513
High         863
Critical     765
Name: RiskCategory, dtype: int64

Attrition rate & avg prob by risk category:
              AttritionBinary  AttritionProb
RiskCategory                                
Critical                0.546          0.572
High                    0.460          0.379
Low                     0.069          0.109
Medium                  0.260          0.243


In [87]:
# Final persona check using model predictions
ideal = df_scored[
    (df_scored["Gender"]           == "Female")  &
    (df_scored["MaritalStatus"]    == "Married") &
    (df_scored["Location"]         == "WFH")     &
    (df_scored["PromotionsLast3Yrs"] >= 1)       &
    (df_scored["AwardsReceived"]   >= 1)         &
    (df_scored["JobSatisfaction"]  >= 3)         &
    (df_scored["OverTime"]         == "No")
]
risky = df_scored[
    (df_scored["MaritalStatus"]   == "Single")                    &
    (df_scored["Department"]      == "Sales")                     &
    (df_scored["MarketRateRatio"] < 0.85)                         &
    (df_scored["OverTime"]        == "Yes")                       &
    (df_scored["ShiftType"].isin(["Night","Rotational"]))
]

print("=== FINAL MODEL PERSONA CHECK ===")
print(f"\n Married WFH Woman, Promoted, Awards, No OT")
print(f"   Model avg: {ideal['AttritionProb'].mean():.1%}  (target: ~5–10%)")
print(f"\nSingle Sales, Underpaid, OT, Night/Rotational Shift")
print(f"   Model avg: {risky['AttritionProb'].mean():.1%}  (target: ~40–55%)")
print(f"\nCompany Average")
print(f"   Model avg: {df_scored['AttritionProb'].mean():.1%}  (target: ~20%)")

=== FINAL MODEL PERSONA CHECK ===

 Married WFH Woman, Promoted, Awards, No OT
   Model avg: 11.9%  (target: ~5–10%)

Single Sales, Underpaid, OT, Night/Rotational Shift
   Model avg: 40.1%  (target: ~40–55%)

Company Average
   Model avg: 26.7%  (target: ~20%)


---
## 11. Save Artifacts

Artifacts keep the same filenames, so your Streamlit app can continue reading them without large changes.


In [88]:
OUT = "artifacts"

# Remove stale folder — avoids Windows permission errors on locked files
if os.path.exists(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)

print(f"📁 Saving to {OUT}/...")

📁 Saving to artifacts/...


In [89]:
# CSVs
df.to_csv(f"{OUT}/Microland_employees.csv", index=False)
df_scored.to_csv(f"{OUT}/employees_scored.csv", index=False)
print("✅ CSVs saved")

✅ CSVs saved


In [90]:
# Save calibrated deployed model + preprocessing artifacts
pickle.dump(calibrated_model, open(f"{OUT}/best_model.pkl", "wb"))
pickle.dump(scaler,           open(f"{OUT}/scaler.pkl",     "wb"))
pickle.dump(encoders,         open(f"{OUT}/encoders.pkl",   "wb"))

print("✅ Model artifacts saved")


✅ Model artifacts saved


In [91]:
meta = {
    "best_model_name"      : best_name,
    "deployed_model_name"  : f"Calibrated {best_name}",
    "is_calibrated_model"  : True,
    "needs_scale"          : needs_scale,
    "feature_names"        : feature_names,
    "drop_cols"            : DROP_COLS,
    "categorical_cols"     : CATEGORICAL,
    "decision_threshold"   : DECISION_THRESHOLD,
    "risk_thresholds"      : {"Critical": 0.45, "High": 0.32, "Medium": 0.18},
    "test_auc"             : round(calibrated_results["test_auc"],    4),
    "test_f1"              : round(calibrated_results["test_f1"],     4),
    "test_acc"             : round(calibrated_results["test_acc"],    4),
    "test_recall"          : round(calibrated_results["test_recall"], 4),
    "cv_auc"               : round(results[best_name]["cv_auc_mean"], 4),
    "all_models": {
        n: {
            "cv_auc"      : round(r["cv_auc_mean"], 4),
            "test_auc"    : round(r["test_auc"],    4),
            "test_f1"     : round(r["test_f1"],     4),
            "test_recall" : round(r["test_recall"], 4),
            "test_acc"    : round(r["test_acc"],    4),
        } for n, r in results.items()
    },
    "dataset_size"         : len(df),
    "trained_on"           : datetime.now().isoformat(),
}
json.dump(meta, open(f"{OUT}/model_meta.json", "w"), indent=2)

print("✅ Metadata JSON saved")
print("\n📦 Artifacts:", os.listdir(OUT))


✅ Metadata JSON saved

📦 Artifacts: ['best_model.pkl', 'employees_scored.csv', 'encoders.pkl', 'Microland_employees.csv', 'model_meta.json', 'scaler.pkl']


In [92]:
print("=" * 56)
print(f"  🏆 BASE WINNER     : {best_name}")
print(f"  🚀 DEPLOYED MODEL  : Calibrated {best_name}")
print("=" * 56)
print(f"  Test AUC  : {meta['test_auc']}")
print(f"  Test F1   : {meta['test_f1']}")
print(f"  Recall    : {meta['test_recall']}")
print(f"  Dataset   : {meta['dataset_size']:,} employees")
print(f"  Features  : {len(feature_names)}")
print("=" * 56)
print("\n▶  streamlit run app.py")


  🏆 BASE WINNER     : Logistic Regression
  🚀 DEPLOYED MODEL  : Calibrated Logistic Regression
  Test AUC  : 0.747
  Test F1   : 0.4487
  Recall    : 0.3933
  Dataset   : 5,000 employees
  Features  : 61

▶  streamlit run app.py


---
## 📖 Design Reference

### Probability Range Targets

| Employee Profile | Risk | Why |
|---|---|---|
| Married WFH woman, promoted, awards, no OT | **~5–8%** | All retention factors stacked |
| P6 senior, 15yr tenure, well paid, day shift | **~6–9%** | Veteran + seniority |
| Average employee | **~20%** | Company baseline |
| P1A fresher, 1yr tenure, single, avg satisfaction | **~30–35%** | New joiner + entry band |
| Single sales, underpaid, OT, night shift | **~45–55%** | All risk factors stacked |

### Top 10 Features (Business View)

| Rank | Feature | Business Meaning |
|------|---------|------------------|
| 1 | `MarketRateRatio` | Are we paying market rate? |
| 2 | `OverTime` | Burnout — strongest single predictor |
| 3 | `JobSatisfaction` + `WorkLifeBalance` | Employee engagement |
| 4 | `StagnationIndex` | Career growth blocked? |
| 5 | `ManagerRating` | People management quality |
| 6 | `BusinessTravel` | Lifestyle impact |
| 7 | `ShiftType` | Night/Rotational = physical strain |
| 8 | `WFH_DaysPerWeek` | Flexibility is a retention lever |
| 9 | `YearsAtCompany` | First year = highest risk |
| 10 | `AwardsReceived` | Recognition gap |